# Modul 1 - Preprocessing Teks
**Praktikum Step 1 (Preprocessing Teks), STKI**

| Keterangan | Isi |
|---|---|
| Data | 30 dokumen (D01 sampai D30) dari `data/korpus.csv` |
| Alat | Python, pandas, Sastrawi |

## Cara menjalankan
1. Di panel file Colab (ikon folder di kiri), buat folder bernama `data`.
2. Unggah `korpus.csv` ke dalam folder `data`.
3. Jalankan sel secara berurutan dari atas ke bawah.

## Pendahuluan
Modul ini membangun tahap pemrosesan teks (*preprocessing*) untuk korpus berita berbahasa Indonesia. Tahapannya adalah pemuatan korpus, *case folding*, tokenisasi dengan ekspresi reguler, penghapusan *stopword* memakai daftar Sastrawi, dan *stemming* Sastrawi dengan cache. Ketiganya digabung dalam fungsi `preprocess()`. Fungsi ini nanti dipakai ulang pada modul indexing.

In [ ]:
!pip install Sastrawi -q

import re
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

## Langkah 1 - Memuat korpus
Korpus dibaca dari `data/korpus.csv`. Judul dan isi digabung menjadi satu teks per dokumen.

In [ ]:
print("[Langkah 1] Memuat korpus")
DATA_PATH = Path("data/korpus.csv")
korpus = pd.read_csv(DATA_PATH)
print("Jumlah dokumen:", len(korpus))
print(korpus[["doc_id", "judul"]].head().to_string(index=False))

# Judul dan isi digabung menjadi satu teks dokumen
korpus["teks"] = korpus["judul"] + ". " + korpus["isi"]
contoh = korpus.loc[16, "teks"]
print("\nTeks contoh (D17):")
print(contoh)

## Langkah 2 - Case folding
Semua huruf diubah menjadi huruf kecil agar "Pertanian" dan "pertanian" dianggap sama.

In [ ]:
print("[Langkah 2] Case folding")

def case_folding(teks):
    return teks.lower()

print(case_folding(contoh))

## Langkah 3 - Tokenisasi
Setiap rangkaian huruf dan angka menjadi satu token. Tanda baca dan spasi menjadi pemisah.

In [ ]:
print("[Langkah 3] Tokenisasi")

def tokenize(teks):
    return re.findall(r"[a-z0-9]+", teks.lower())

token_contoh = tokenize(case_folding(contoh))
print(f"Jumlah token: {len(token_contoh)}")
print(token_contoh)

# Pembanding: split() tidak membuang tanda baca
print("Pembanding split():", case_folding(contoh).split()[:12])

## Langkah 4 - Penghapusan stopword
Daftar stopword bahasa Indonesia diambil dari Sastrawi.

In [ ]:
print("[Langkah 4] Penghapusan stopword")
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

STOPWORDS = set(StopWordRemoverFactory().get_stop_words())
print("Jumlah stopword:", len(STOPWORDS))
print("40 stopword pertama (urut abjad):", sorted(STOPWORDS)[:40])

def remove_stopwords(tokens, stopwords=STOPWORDS):
    return [t for t in tokens if t not in stopwords]

tanpa_sw = remove_stopwords(token_contoh)
print(f"Jumlah token: {len(token_contoh)} -> {len(tanpa_sw)}")
print(tanpa_sw)

## Langkah 5 - Stemming
Kata berimbuhan diubah menjadi kata dasar. Hasil stemming disimpan dalam cache, sehingga setiap kata unik hanya di-stem satu kali.

In [ ]:
print("[Langkah 5] Stemming")
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

stemmer = StemmerFactory().create_stemmer()
for kata in ["pembelajaran", "pertanian", "petani", "pencarian", "keamanan",
             "memantau", "berolahraga", "perekonomian", "mahasiswa"]:
    print(f"  {kata:<14} -> {stemmer.stem(kata)}")

_cache = {}

def stem(token):
    if token not in _cache:
        _cache[token] = stemmer.stem(token)
    return _cache[token]

print("Hasil stemming pada contoh:")
print(" ".join(stem(t) for t in tanpa_sw))

## Langkah 6 - Menggabungkan menjadi fungsi `preprocess()`
Empat tahap digabung dalam satu fungsi. Parameter `use_stopword` dan `use_stemming` memungkinkan tahap tertentu dimatikan.

In [ ]:
print("[Langkah 6] Fungsi preprocess")

def preprocess(teks, use_stopword=True, use_stemming=True):
    tokens = tokenize(case_folding(teks))
    if use_stopword:
        tokens = remove_stopwords(tokens)
    if use_stemming:
        tokens = [stem(t) for t in tokens]
    return tokens

korpus["tokens"] = korpus["teks"].apply(preprocess)
ringkas = pd.DataFrame({
    "doc_id": korpus["doc_id"],
    "jumlah_token": korpus["tokens"].str.len(),
    "8 token pertama": korpus["tokens"].apply(lambda t: " ".join(t[:8])),
})
print(ringkas.head().to_string(index=False))

## Langkah 7 - Mengukur pengaruh tiap tahap
Jumlah token dan ukuran kosakata dihitung untuk setiap kombinasi tahap.

In [ ]:
print("[Langkah 7] Pengaruh tiap tahap")
tahap = {
    "Tokenisasi saja": dict(use_stopword=False, use_stemming=False),
    "+ Stopword removal": dict(use_stopword=True, use_stemming=False),
    "+ Stemming (tanpa stopword removal)": dict(use_stopword=False, use_stemming=True),
    "+ Stopword removal + Stemming": dict(use_stopword=True, use_stemming=True),
}
baris = []
for nama, opsi in tahap.items():
    semua = [t for teks in korpus["teks"] for t in preprocess(teks, **opsi)]
    baris.append({"Tahap": nama, "Jumlah token": len(semua), "Ukuran kosakata": len(set(semua))})
print(pd.DataFrame(baris).to_string(index=False))

print("\nFrekuensi term teratas (setelah preprocessing lengkap):")
frekuensi = Counter(t for tokens in korpus["tokens"] for t in tokens)
print(pd.DataFrame(frekuensi.most_common(15), columns=["term", "frekuensi"]).to_string(index=False))

## Langkah 8 - Memastikan fungsi identik dengan `stki_utils.py`
Jika berkas `stki_utils.py` ikut diunggah, fungsi `preprocess()` di sini dibandingkan dengan versi di berkas tersebut. Jika tidak diunggah, langkah ini dilewati.

In [ ]:
print("[Langkah 8] Pencocokan dengan stki_utils.py")
try:
    import stki_utils
    cocok = all(preprocess(t) == stki_utils.preprocess(t) for t in korpus["teks"])
    print("Hasil identik dengan stki_utils.preprocess:", cocok)
except ImportError:
    print("stki_utils.py tidak diunggah. Langkah pencocokan dilewati.")

---
# Tugas Step 1

## Tugas 1 - Overstemming dan understemming
Stem yang menaungi lebih dari satu kata asli dikumpulkan sebagai kandidat.

In [ ]:
print("[Tugas 1] Kandidat overstemming")

kata_asli = {t for teks in korpus["teks"]
             for t in remove_stopwords(tokenize(case_folding(teks)))}
kelompok = defaultdict(set)
for kata in kata_asli:
    kelompok[stem(kata)].add(kata)

gabungan = {s: sorted(k) for s, k in kelompok.items() if len(k) > 1}
print(f"Jumlah stem yang menaungi lebih dari satu kata: {len(gabungan)}")
for s, daftar in sorted(gabungan.items()):
    print(f"  {s:<12} <- {', '.join(daftar)}")

print("\nPengecekan bentuk kata tertentu:")
for kata in ["melalui", "pembelajaran", "tokenisasi", "pemrosesan", "imbuhan", "berimbuhan"]:
    print(f"  {kata:<14} -> {stem(kata)}")

mentah = [t for teks in korpus["teks"] for t in tokenize(case_folding(teks))]
print("\nBentuk mentah 'melalui':", mentah.count("melalui"), "| 'lalu':", mentah.count("lalu"))

### Jawaban Tugas 1
**Overstemming** (kata berbeda makna menjadi stem yang sama):
- **melalui -> lalu.** "Melalui" berarti perantara, sedangkan "lalu" berarti waktu. Bentuk mentah "melalui" muncul 8 kali dan "lalu" 1 kali, sehingga stem "lalu" berjumlah 9. Pada D01, "ditularkan melalui gigitan" menjadi "tular lalu gigit". Kueri tentang waktu dapat mengambil dokumen yang memuat "melalui" dalam arti cara. Presisi turun.
- **meningkat dan tingkat -> tingkat.** "Meningkat" berarti naik, sedangkan "tingkat" berarti level atau taraf. Stem "tingkat" berjumlah 6 pada Langkah 7, sehingga kueri tentang kenaikan dapat mencocokkan dokumen yang hanya memuat "tingkat" dalam arti level.
- **buatan dan membuat -> buat.** Frasa "kecerdasan buatan" menjadi kurang spesifik, karena dokumen yang memuat "membuat" (*make*) ikut cocok dengan kata "buatan".

**Understemming** (makna sama tetapi stem berbeda):
- **pemrosesan** tetap utuh (2 kemunculan), tidak tergabung dengan "proses" atau "memproses". Kueri "proses" tidak menemukan dokumen yang memuat "pemrosesan".
- **tokenisasi** dan **stemming** tetap utuh, tidak terhubung dengan "token" atau "stem".

Pasangan yang tergabung dengan benar: "pertanian" dan "petani" menjadi "tani", "imbuhan" dan "berimbuhan" menjadi "imbuh", "berolahraga" dan "olahraga" menjadi "olahraga", serta "kesehatan" dan "sehat" menjadi "sehat".

**Dampak:** overstemming menurunkan presisi, karena dokumen yang hanya cocok secara bentuk kata ikut terambil. Understemming menurunkan recall, karena bentuk lain dari akar yang sama tidak ikut terambil.

## Tugas 2 - Stopword khusus domain
Ditambahkan dua kata, "dapat" dan "membantu", lalu diukur pengaruhnya terhadap kosakata.

In [ ]:
print("[Tugas 2] Stopword khusus domain")
STOPWORDS_DOMAIN = STOPWORDS | {"dapat", "membantu"}

def ukuran_kosakata(stopwords):
    return len({stem(t) for teks in korpus["teks"]
                for t in remove_stopwords(tokenize(case_folding(teks)), stopwords)})

print("Kosakata sebelum:", ukuran_kosakata(STOPWORDS))
print("Kosakata sesudah:", ukuran_kosakata(STOPWORDS_DOMAIN))
print("Kemunculan bentuk mentah 'membantu':", mentah.count("membantu"))
print("Bentuk mentah lain yang mengandung 'bantu':",
      sorted({t for t in mentah if "bantu" in t and t != "membantu"}))

### Jawaban Tugas 2
"Dapat" sudah berada di daftar stopword Sastrawi, sehingga penambahannya tidak mengubah ukuran kosakata.

"Membantu" berbeda. Stopword dicek sebelum stemming, sehingga bentuk mentah "membantu" harus ada di daftar agar terhapus. Bentuk "membantu" muncul 8 kali dan tidak ada bentuk mentah lain yang mengandung "bantu", sehingga seluruh 8 kemunculan stem "bantu" berasal dari "membantu".

| Daftar stopword | Jumlah token | Ukuran kosakata |
|---|---|---|
| Sastrawi (bawaan) | 863 | 472 |
| Sastrawi + dapat, membantu | 855 | 471 |
| Selisih | 8 | 1 |

Karena stopword diperiksa pada token mentah, bentuk yang ditambahkan harus sama dengan bentuk di korpus, bukan stem-nya.

## Tugas 3 - Tokenizer dengan tanda hubung
Pola baru `[a-z0-9]+(?:-[a-z0-9]+)*` menjaga kata ulang seperti "anak-anak" tetap menjadi satu token.

In [ ]:
print("[Tugas 3] Tokenizer dengan tanda hubung")

def tokenize_hubung(teks):
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", teks.lower())

kalimat_uji = "Anak-anak bermain di kebun sayur-mayur, bukan di jalan raya."
print("Kalimat uji        :", kalimat_uji)
print("Tokenizer baru     :", tokenize_hubung(case_folding(kalimat_uji)))
print("Tokenizer lama     :", tokenize(case_folding(kalimat_uji)))

### Jawaban Tugas 3
Pola `[a-z0-9]+(?:-[a-z0-9]+)*` menjaga "anak-anak" dan "sayur-mayur" tetap satu token. Tokenizer lama memecahnya menjadi dua token.

Pada korpus ini tidak ada token yang mengandung tanda hubung, sehingga jumlah token tetap 1033 di kedua tokenizer. Tokenizer baru tidak mengubah hasil Step 1 pada korpus ini, tetapi tetap berguna untuk teks lain yang memuat kata ulang. Hasil stemming untuk token yang mengandung tanda hubung perlu dicek secara terpisah, karena stemmer dirancang untuk kata tanpa tanda hubung.

## Tugas 4 - Tahap yang paling mengurangi kosakata
Pengurangan kosakata dihitung dari tabel Langkah 7.

In [ ]:
print("[Tugas 4] Pengurangan kosakata per tahap")
tabel = pd.DataFrame(baris).set_index("Tahap")
dasar = tabel.loc["Tokenisasi saja", "Ukuran kosakata"]
print(tabel.assign(Pengurangan=lambda d: dasar - d["Ukuran kosakata"]).to_string())

### Jawaban Tugas 4
Stemming adalah tahap yang paling banyak mengurangi ukuran kosakata, dari 574 menjadi 501 (berkurang 73 atau 12,7%). Stopword removal hanya menurunkan kosakata menjadi 541 (berkurang 33 atau 5,7%). Gabungan keduanya menghasilkan kosakata 472 (berkurang 102 atau 17,8%).

| Tahap | Kurang kosakata | % kosakata | Kurang token | % token |
|---|---|---|---|---|
| Stopword removal saja | 33 | 5,7 | 170 | 16,5 |
| Stemming saja | 73 | 12,7 | 0 | 0,0 |
| Stopword removal + Stemming | 102 | 17,8 | 170 | 16,5 |

**Penyebabnya:** stemming menyatukan banyak bentuk berimbuhan yang berbeda menjadi satu kata dasar, misalnya "pembelajaran" menjadi "ajar" dan "pembayaran" menjadi "bayar". Stopword removal menghapus banyak token (1033 menjadi 863), tetapi kata stopword yang sama muncul berulang kali. Kosakata menghitung setiap jenis kata hanya sekali, sehingga penghapusan token berulang tidak banyak mengurangi jumlah jenis kata.

---
## Kesimpulan
- Preprocessing menurunkan kosakata dari 574 menjadi 472 term. Stemming memberi pengurangan terbesar pada kosakata, sedangkan stopword removal memberi pengurangan terbesar pada jumlah token. Kedua tahap ini membawa risiko over dan understemming yang berdampak pada presisi dan recall.